In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import Libraries 

In [2]:
# Libraries for data obtaining
from pathlib import Path

In [3]:
#Download processed dataset from Zenodo
!wget -O /kaggle/working/data.tar.gz "https://zenodo.org/records/2652278/files/data.tar.gz?download=1"

--2026-04-27 12:08:06--  https://zenodo.org/records/2652278/files/data.tar.gz?download=1
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.184.98.114, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 539905064 (515M) [application/octet-stream]
Saving to: ‘/kaggle/working/data.tar.gz’

/kaggle/working/dat 100%[===================>] 514.89M  27.1MB/s    in 20s     

2026-04-27 12:08:27 (25.6 MB/s) - ‘/kaggle/working/data.tar.gz’ saved [539905064/539905064]



In [4]:
#Extract data
!tar -xzf /kaggle/working/data.tar.gz -C /kaggle/working/

In [5]:
root = Path("/kaggle/working/data")
 
# Look for directories that contain train.csv/valid.csv/test.csv
all_dirs = sorted([d for d in root.rglob("*") if d.is_dir()])
cell_type_dirs = sorted([d for d in all_dirs if (d / "train.csv").exists()])
print(f"  Cell type directories found: {len(cell_type_dirs)}")

  Cell type directories found: 56


In [6]:
#Inspecting one cell type in detail
e047_dir = next((d for d in cell_type_dirs if d.name == "E047"), cell_type_dirs[0])
print(f"  Path: {e047_dir}")
 
splits = {}
for split in ["train", "valid", "test"]:
    fpath = e047_dir / f"{split}.csv"
    df = pd.read_csv(fpath, header=None)
    splits[split] = df
    print(f"\n  {split}.csv:")
    print(f"    Shape:   {df.shape}")
    print(f"    Columns: {df.shape[1]} total")
    print(f"    First row (first 10 values): {df.iloc[0, :10].tolist()}")
    print(f"    Last column (label?): unique values = {sorted(df.iloc[:, -1].unique())}")
    print(f"    Second-to-last col sample: {df.iloc[:3, -2].tolist()}")
 

  Path: /kaggle/working/data/E003/classification

  train.csv:
    Shape:   (660100, 8)
    Columns: 8 total
    First row (first 10 values): [3, 1, 0, 1, 0, 2, 0, 1]
    Last column (label?): unique values = [np.int64(0), np.int64(1)]
    Second-to-last col sample: [0, 0, 1]

  valid.csv:
    Shape:   (660100, 8)
    Columns: 8 total
    First row (first 10 values): [132963, 1, 0, 0, 2, 2, 0, 1]
    Last column (label?): unique values = [np.int64(0), np.int64(1)]
    Second-to-last col sample: [0, 1, 0]

  test.csv:
    Shape:   (660000, 8)
    Columns: 8 total
    First row (first 10 values): [172936, 1, 0, 6, 1, 2, 0, 0]
    Last column (label?): unique values = [np.int64(0), np.int64(1)]
    Second-to-last col sample: [0, 2, 3]


In [7]:
splits['train'].head()

,0,1,2,3,4,5,6,7
0,3,1,0,1,0,2,0,1
1,3,2,1,2,2,8,0,1
2,3,3,0,5,2,6,1,1
3,3,4,1,2,5,2,1,1
4,3,5,2,2,1,16,1,1


In [8]:
splits['train'][7].value_counts()

7
0    425200
1    234900
Name: count, dtype: int64

##### Dataset columns are:
- GeneID
- Bin ID
- H3K27me3 count
- H3K36me3 count
- H3K4me1 count
- H3K4me3 count
- H3K9me3 counts
- Binary Label for gene expression (0/1)
